In [20]:
import os
# Disable GPU visibility
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Disable XLA (this is the critical fix)
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")
# Disable XLA JIT explicitly
tf.config.optimizer.set_jit(False)

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import set_random_seed
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

Read YOLO data & process

In [26]:
# DATA PATH
current_dir = Path(os.getcwd())
data_path = current_dir.parent / "Dataset"
if not data_path.exists():
    raise FileNotFoundError(f"Cannot find Dataset. Please check path: {data_path}")

# TODO：For MARS or For MOON
path = os.path.join(data_path,'Moon')
print(path)

/Users/jessie_guo/Downloads/ML Intern - Crater/Dataset/Moon


In [27]:
train_img_path = os.path.join(path, 'images', 'train')
train_lbl_path = os.path.join(path, 'labels', 'train')

valid_img_path = os.path.join(path, 'images', 'val')
valid_lbl_path = os.path.join(path, 'labels', 'val')

test_img_path = os.path.join(path, 'images', 'test')
test_lbl_path = os.path.join(path, 'labels', 'test')

IMG_SIZE = 128
NUM_CLASSES = 3

# path to save processed images
processed_path = os.path.join(path, 'processed')
os.makedirs(processed_path, exist_ok=True)

In [28]:
# process a dataset by extracting craters from images based on YOLO-format labels and saving them in class-specific directories. 
def process_dataset(img_dir, lbl_dir, output_dir):
    for img_file in os.listdir(img_dir):
        if not img_file.endswith('.jpg'):
            continue # skip non-image files
            
        # Get the corresponding label file
        base_name = os.path.splitext(img_file)[0]
        lbl_file = os.path.join(lbl_dir, f"{base_name}.txt")
        
        # Process single image
        img = Image.open(os.path.join(img_dir, img_file))
        img_w, img_h = img.size
        
        with open(lbl_file, 'r') as f:
            for idx, line in enumerate(f.readlines()):
                class_id, xc, yc, w, h = map(float, line.strip().split())
                # Ensure image bounds
                x1 = int((xc - w/2) * img_w)
                y1 = int((yc - h/2) * img_h)
                x2 = int((xc + w/2) * img_w)
                y2 = int((yc + h/2) * img_h)
                                
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(img_w, x2), min(img_h, y2)
                
                # Crop the crater and resize it
                crater = img.crop((x1, y1, x2, y2))
                crater = crater.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
                # save
                class_dir = os.path.join(output_dir, str(int(class_id)))
                os.makedirs(class_dir, exist_ok=True)
                crater.save(os.path.join(class_dir, f"{base_name}_{idx}.jpg"))

# process all dataset
process_dataset(train_img_path, train_lbl_path, os.path.join(processed_path, 'train'))
process_dataset(valid_img_path, valid_lbl_path, os.path.join(processed_path, 'val'))
process_dataset(test_img_path, test_lbl_path, os.path.join(processed_path, 'test'))

In [29]:
# define a PyTorch dataset for loading crater images and their corresponding labels.
class CraterDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data = []
        self.transform = transform
        
        for class_id in range(NUM_CLASSES):
            class_dir = os.path.join(data_dir, str(class_id))
            for img_file in os.listdir(class_dir):
                self.data.append((os.path.join(class_dir, img_file), class_id))
                
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = Image.open(img_path).convert('RGB') 
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

In [30]:
# Define data enhancement
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Loading data from category folder
train_generator = train_datagen.flow_from_directory(
    os.path.join(processed_path, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical',
    color_mode='grayscale'
)
val_generator = train_datagen.flow_from_directory(
    os.path.join(processed_path, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical',
    color_mode='grayscale'
)

Found 1149 images belonging to 3 classes.
Found 320 images belonging to 3 classes.


In [31]:
# test data
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    os.path.join(processed_path, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical', 
    shuffle=False,
    color_mode='grayscale'
)

Found 671 images belonging to 3 classes.


Model Test & Prediction

CNN Model

In [32]:
def create_model():
    model = Sequential()
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        MaxPooling2D((2,2)),

        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),

        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D((2,2)),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [33]:
# Parameter settings
NUM_RUNS = 30
class_names = list(test_generator.class_indices.keys())

# Initialize storage structure for metrics
metrics = {
    'classes': {cls: {'precision': [], 'recall': [], 'f1': []} for cls in class_names},
    'macro_avg': {'precision': [], 'recall': [], 'f1': []},
    'weighted_avg': {'precision': [], 'recall': [], 'f1': []},
    'accuracy': []
}

for run in range(NUM_RUNS):
    print(f"\n=========== Training Run {run+1}/{NUM_RUNS} ===========")
    
    # Reset model and random seeds before each training run
    tf.keras.backend.clear_session()
    np.random.seed(run)
    tf.random.set_seed(run)
    
    # Create a new model
    model = create_model()
    
    # Define callbacks
    checkpoint = ModelCheckpoint('best_model.keras', 
                                monitor='val_loss',
                                save_best_only=True,
                                mode='min',
                                verbose=0)
    early_stop = EarlyStopping(monitor='val_loss',
                              patience=5,
                              restore_best_weights=True)
    
    # Train the model
    history = model.fit(
        train_generator,
        epochs=30,
        validation_data=val_generator,
        callbacks=[checkpoint, early_stop],
        verbose=1  # Adjust verbosity as needed
    )
    
    # Load the best model
    best_model = load_model('best_model.keras')
    
    # Predict on the test set
    y_pred = best_model.predict(test_generator)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = test_generator.classes
    
    # Generate classification report
    report = classification_report(y_true, y_pred_classes, 
                                 target_names=class_names,
                                 output_dict=True)
    
    # Collect metrics for each class
    for cls in class_names:
        metrics['classes'][cls]['precision'].append(report[cls]['precision'])
        metrics['classes'][cls]['recall'].append(report[cls]['recall'])
        metrics['classes'][cls]['f1'].append(report[cls]['f1-score'])
    
    # Collect macro and weighted averages
    metrics['macro_avg']['precision'].append(report['macro avg']['precision'])
    metrics['macro_avg']['recall'].append(report['macro avg']['recall'])
    metrics['macro_avg']['f1'].append(report['macro avg']['f1-score'])
    
    metrics['weighted_avg']['precision'].append(report['weighted avg']['precision'])
    metrics['weighted_avg']['recall'].append(report['weighted avg']['recall'])
    metrics['weighted_avg']['f1'].append(report['weighted avg']['f1-score'])
    
    # Collect accuracy
    metrics['accuracy'].append(report['accuracy'])

# PR Curves
precision = dict()
recall = dict()
binarise_y = label_binarize(y_true, classes=[*range(NUM_CLASSES)])

mean_recall = np.linspace(0, 1, 100)
precisions = []

for i in range(NUM_CLASSES):
    precision, recall, _ = precision_recall_curve(binarise_y[:, i],
                                                        y_pred[:, i])
    precisions.append(np.interp(mean_recall, recall[::-1], precision[::-1]))
    plt.plot(recall, precision, lw=2, label='Class {}'.format(i))

mean_precision = np.mean(precisions, axis=0)
plt.plot(mean_recall, mean_precision, lw = 2, label = 'All Classes')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="best")
# TODO：For MARS or For MOON
plt.title(f"Precision Recall Curves of CNN for Moon")
plt.tight_layout()
plt.savefig(f"PR Curves of CNN Moon.png")
plt.close()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title('Training Losses of CNN for Moon')
plt.savefig('Training Losses CNN Moon.png')
plt.close()


=========== Training Run 1/30 ===========
Epoch 1/30


/opt/anaconda3/envs/crater/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 268ms/step - accuracy: 0.9269 - loss: 0.3357 - val_accuracy: 0.9500 - val_loss: 0.1799
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 273ms/step - accuracy: 0.9399 - loss: 0.1858 - val_accuracy: 0.9531 - val_loss: 0.1731
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 253ms/step - accuracy: 0.9513 - loss: 0.1817 - val_accuracy: 0.9531 - val_loss: 0.1272
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 250ms/step - accuracy: 0.9600 - loss: 0.1369 - val_accuracy: 0.9719 - val_loss: 0.0951
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 251ms/step - accuracy: 0.9661 - loss: 0.1073 - val_accuracy: 0.9719 - val_loss: 0.0762
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 250ms/step - accuracy: 0.9695 - loss: 0.1032 - val_accuracy: 0.9594 - val_loss: 0.1158
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 249ms/step - accuracy: 0.9669 - loss: 0.1101 - val_accuracy: 0.9625 - val_loss: 0.1236
Epoch 8/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 250ms/step - accuracy: 0.9704 - loss: 0.1124 - val_accuracy: 0.9656 - va

/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

Epoch 1/30


...

/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

Epoch 1/30


/opt/anaconda3/envs/crater/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 261ms/step - accuracy: 0.9312 - loss: 0.3433 - val_accuracy: 0.9500 - val_loss: 0.2212
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 249ms/step - accuracy: 0.9443 - loss: 0.2339 - val_accuracy: 0.9500 - val_loss: 0.1814
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 248ms/step - accuracy: 0.9469 - loss: 0.1851 - val_accuracy: 0.9500 - val_loss: 0.1554
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 248ms/step - accuracy: 0.9426 - loss: 0.1789 - val_accuracy: 0.9563 - val_loss: 0.1250
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 250ms/step - accuracy: 0.9539 - loss: 0.1428 - val_accuracy: 0.9656 - val_loss: 0.1221
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 248ms/step - accuracy: 0.9591 - loss: 0.1518 - val_accuracy: 0.9594 - val_loss: 0.1461
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 247ms/step - accuracy: 0.9661 - loss: 0.1108 - val_accuracy: 0.9563 - val_loss: 0.1515
Epoch 8/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 247ms/step - accuracy: 0.9600 - loss: 0.1322 - val_accuracy: 0.9656 - val

/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/crater/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [ ]:
print("\n\n=== Classification Report with Mean±Std ===")

# for class
max_class_length = max(len(str(cls)) for cls in class_names)
print(f"\n{'Class':<{max_class_length}} {'Precision':<6} {'Recall':<6} {'F1-score':<6}  Support")
for idx, cls in enumerate(class_names):
    prec_mean = np.mean(metrics['classes'][cls]['precision'])
    prec_std = np.std(metrics['classes'][cls]['precision'])
    rec_mean = np.mean(metrics['classes'][cls]['recall'])
    rec_std = np.std(metrics['classes'][cls]['recall'])
    f1_mean = np.mean(metrics['classes'][cls]['f1'])
    f1_std = np.std(metrics['classes'][cls]['f1'])
    
    print(f"{idx:<{max_class_length}} "
          f"{prec_mean:.2f}±{prec_std:.2f}  "
          f"{rec_mean:.2f}±{rec_std:.2f}  "
          f"{f1_mean:.2f}±{f1_std:.2f}  "
          f"{test_generator.classes.tolist().count(idx)}")
# for all
def print_avg_row(name, metric_dict):
    prec = f"{np.mean(metric_dict['precision']):.2f}±{np.std(metric_dict['precision']):.2f}"
    rec = f"{np.mean(metric_dict['recall']):.2f}±{np.std(metric_dict['recall']):.2f}"
    f1 = f"{np.mean(metric_dict['f1']):.2f}±{np.std(metric_dict['f1']):.2f}"
    print(f"{name:<{max_class_length}} {prec:<12} {rec:<12} {f1:<12} ")

total_samples = len(test_generator.classes)
print(f"\n{'accuracy':<{max_class_length}} {'':<12} {'':<12} "
      f"{np.mean(metrics['accuracy']):.2f}±{np.std(metrics['accuracy']):.2f}  "
      f"{total_samples}")
print_avg_row('macro avg', metrics['macro_avg'])
print_avg_row('weighted avg', metrics['weighted_avg'])